# Magic Hour · REFace Face Swap

Internal demo for **diffusion face swapping** (WACV 2025) — not Krea2 image editing.

| Input | Role |
| --- | --- |
| **Body** | Scene — pose, expression, clothing, background |
| **Face** | Identity — face to transfer |

Upstream: [Sanoojan/REFace](https://github.com/Sanoojan/REFace)

**Recommended workflow:** edit §1 → **Runtime → Run all**.

**Expected runtime (warm, after models loaded)**

| GPU | Typical wall time |
| --- | --- |
| **A100** (40 GB) | ~1–2 min / image |
| **T4** (16 GB) | ~3–6 min / image |

Cold start downloads ~7 GB (Drive-cached). Prefer **A100**.

Warm re-run: change knobs or re-upload → re-run **§5**.


---
## 1 · Settings

Only these knobs are meant to change.


In [ ]:
# === User-facing knobs ===
SEED = 42
STEPS = 50                    # DDIM steps
CFG = 3.5                     # guidance / scale
OUTPUT_LONG_SIDE = 1024       # body long side before swap (px); 0 = native
DEBUG = False

# Multi-person body photos: which face to swap
# largest | rightmost | leftmost | index (use BODY_FACE_INDEX)
BODY_FACE_POLICY = "largest"
BODY_FACE_INDEX = 0

ENABLE_MULTI_FACE_UI = False

IDENTITY_THRESH = 0.35
BODY_PSNR_THRESH = 28.0
PINNED_COMMIT = None

print("Settings")
print(f"  SEED={SEED}  STEPS={STEPS}  CFG={CFG}  OUTPUT_LONG_SIDE={OUTPUT_LONG_SIDE}")
print(f"  DEBUG={DEBUG}")
print(f"  BODY_FACE_POLICY={BODY_FACE_POLICY}  BODY_FACE_INDEX={BODY_FACE_INDEX}")
print(f"  IDENTITY_THRESH={IDENTITY_THRESH}  BODY_PSNR_THRESH={BODY_PSNR_THRESH}")
print(f"  PINNED_COMMIT={PINNED_COMMIT or '(none — use pulled HEAD)'}")


---
## 2 · Setup _(first run / reconnect — collapsed by default)_

Mounts Drive, syncs this repo, clones REFace, downloads Hugging Face checkpoints to Drive.

> First run can take 20–40+ minutes (~7 GB). Later reconnects reuse Drive cache.


In [ ]:
#@title Setup: GPU · Drive · REFace
from pathlib import Path
import importlib.util
import os
import subprocess

assert Path("/content").exists(), "This notebook is for Google Colab (/content)."

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU detected. Runtime → Change runtime type → GPU (prefer A100), then Run all.")
print(f"✓ GPU  {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB)")

from google.colab import drive
print("→ Mounting Google Drive…")
drive.mount("/content/drive")

REPO_URL = "https://github.com/malihashar/headswap_V2.git"
REPO = Path("/content/headswap_V2")
print("→ Syncing headswap_V2…")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)

pin = globals().get("PINNED_COMMIT")
if pin:
    subprocess.run(["git", "fetch", "--depth", "1", "origin", str(pin)], check=False)
    subprocess.run(["git", "checkout", str(pin)], check=False)

!pip install -q -e .

spec = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
colab_env.ensure_import_path(REPO)

os.environ["REFACE_ROOT"] = "/content/REFace"
os.environ["REFACE_DRIVE_ROOT"] = "/content/drive/MyDrive/headswap_reface"
os.environ["HEADSWAP_REPO"] = str(REPO)

print("→ Running REFace bootstrap…")
!bash scripts/setup_reface_colab.sh
print("✓ Setup done.")


---
## 3 · Upload body & face

JPG / PNG / WEBP. Multi-face bodies use **BODY_FACE_POLICY** from §1.


In [ ]:
import importlib.util
from pathlib import Path
from google.colab import files
from IPython.display import display, Markdown

REPO = Path("/content/headswap_V2")
spec_env = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec_env)
spec_env.loader.exec_module(colab_env)
colab_env.ensure_import_path(REPO)

spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

custom = REPO / "data" / "custom"
custom.mkdir(parents=True, exist_ok=True)
cache_dir = REPO / ".cache" / "headswap_v2"
cache_dir.mkdir(parents=True, exist_ok=True)

BODY_FACE_POLICY = str(globals().get("BODY_FACE_POLICY", "largest"))
BODY_FACE_INDEX = int(globals().get("BODY_FACE_INDEX", 0))

try:
    print("Upload BODY image (scene / clothing / pose)…")
    up_body = files.upload()
    if not up_body:
        raise colab_demo.DemoError("No body image uploaded.")
    body_path = custom / "body.png"
    body_im = colab_demo.save_upload(up_body[next(iter(up_body))], body_path)
    for msg in colab_demo.check_image_geometry(body_im, "Body"):
        colab_demo.warn(msg)
    body_face = colab_demo.require_face(body_im, cache_dir, "body")

    print("Upload FACE image (identity donor)…")
    up_face = files.upload()
    if not up_face:
        raise colab_demo.DemoError("No face image uploaded.")
    face_path = custom / "face.png"
    face_im = colab_demo.save_upload(up_face[next(iter(up_face))], face_path)
    for msg in colab_demo.check_image_geometry(face_im, "Face"):
        colab_demo.warn(msg)
    face_face = colab_demo.require_face(face_im, cache_dir, "face")
except colab_demo.DemoError as exc:
    colab_demo.fail(str(exc))
    raise SystemExit(str(exc))

display(Markdown("### Inputs"))
print(f"Body: {body_face['face_count']} face(s), size={body_im.size}, policy={BODY_FACE_POLICY}")
display(body_im.resize((320, 320)))
print(f"Face: {face_face['face_count']} face(s), conf={face_face['confidence']}  size={face_im.size}")
display(face_im.resize((320, 320)))
colab_demo.ok(f"Saved → {body_path} , {face_path}")


---
## 4 · Preflight _(collapsed)_


In [ ]:
#@title Preflight + environment summary
import importlib.util, os
from pathlib import Path
from PIL import Image

REPO = Path("/content/headswap_V2")
REFACE = Path(os.environ.get("REFACE_ROOT", "/content/REFace"))
spec_env = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec_env)
spec_env.loader.exec_module(colab_env)
colab_env.ensure_import_path(REPO)
spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

SEED = int(globals().get("SEED", 42))
STEPS = int(globals().get("STEPS", 50))
CFG = float(globals().get("CFG", 3.5))
OUTPUT_LONG_SIDE = int(globals().get("OUTPUT_LONG_SIDE", 1024))
DEBUG = bool(globals().get("DEBUG", False))
BODY_FACE_POLICY = str(globals().get("BODY_FACE_POLICY", "largest"))
BODY_FACE_INDEX = int(globals().get("BODY_FACE_INDEX", 0))
IDENTITY_THRESH = float(globals().get("IDENTITY_THRESH", 0.35))
BODY_PSNR_THRESH = float(globals().get("BODY_PSNR_THRESH", 28.0))
PINNED_COMMIT = globals().get("PINNED_COMMIT", None)

parse_path = REFACE / "Other_dependencies" / "face_parsing" / "79999_iter.pth"
ckpt_ok = (REFACE / "models/REFace/checkpoints/last.ckpt").exists() or (REFACE / "models/REFace/checkpoints/saved.ckpt").exists()
cfg_ok = (REFACE / "models/REFace/configs/project_ffhq.yaml").exists()
parse_ok = parse_path.is_file() and parse_path.stat().st_size > 1_000_000

# Auto-heal missing face parsing weight (common first-run gap).
if not parse_ok:
    print("→ face parsing weight missing — downloading now…")
    try:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(
            repo_id="Sanoojan/REFace",
            filename="Other_dependencies/face_parsing/79999_iter.pth",
            local_dir=str(Path("/content/drive/MyDrive/headswap_reface/weights/hf")),
            local_dir_use_symlinks=False,
        )
        parse_path.parent.mkdir(parents=True, exist_ok=True)
        import shutil
        if not parse_path.exists():
            try:
                parse_path.symlink_to(p)
            except Exception:
                shutil.copy2(p, parse_path)
    except Exception as exc:
        print(f"HF download failed ({exc}); trying gdown…")
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown
        parse_path.parent.mkdir(parents=True, exist_ok=True)
        gdown.download(id="154JgKpzCPW82qINcVieuPH3fZ2e0P812", output=str(parse_path), quiet=False)
    parse_ok = parse_path.is_file() and parse_path.stat().st_size > 1_000_000

try:
    gpu = colab_demo.verify_gpu()
    VERSIONS = colab_demo.collect_versions(repo=REPO, comfyui=None, pinned_commit=PINNED_COMMIT)
    if not (ckpt_ok and cfg_ok and parse_ok):
        colab_demo.fail(
            f"Missing REFace assets (ckpt={ckpt_ok} cfg={cfg_ok} parsing={parse_ok})."
        )
        print("Fix: re-run §2, or run:")
        print("  !bash /content/headswap_V2/scripts/setup_reface_colab.sh")
        raise RuntimeError(
            f"Missing REFace assets (ckpt={ckpt_ok} cfg={cfg_ok} parsing={parse_ok})"
        )
    cache_dir = REPO / ".cache" / "headswap_v2"
    colab_demo.require_face(Image.open(REPO/"data/custom/body.png").convert("RGB"), cache_dir, "body")
    colab_demo.require_face(Image.open(REPO/"data/custom/face.png").convert("RGB"), cache_dir, "face")
    print("✓ Preflight OK")
    print(f"  GPU={gpu.get('name')}  reface={REFACE}")
    print(f"  SEED={SEED} STEPS={STEPS} CFG={CFG} OUTPUT_LONG_SIDE={OUTPUT_LONG_SIDE}")
    print(f"  BODY_FACE_POLICY={BODY_FACE_POLICY} BODY_FACE_INDEX={BODY_FACE_INDEX}")
    print(f"  face_parsing={parse_path} ({parse_path.stat().st_size} bytes)")
except colab_demo.DemoError as exc:
    colab_demo.fail(str(exc))
    raise RuntimeError(str(exc)) from exc



---
## 5 · Run inference

Each run writes `/content/headswap_outputs/run_YYYYMMDD_HHMMSS/`.


In [ ]:
import importlib.util, os, time, traceback
from pathlib import Path
from PIL import Image

REPO = Path("/content/headswap_V2")
spec_env = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec_env)
spec_env.loader.exec_module(colab_env)
colab_env.ensure_import_path(REPO)
spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

from headswap.config import load_config
from headswap.pipelines import create_pipeline

SEED = int(globals().get("SEED", 42))
STEPS = int(globals().get("STEPS", 50))
CFG = float(globals().get("CFG", 3.5))
OUTPUT_LONG_SIDE = int(globals().get("OUTPUT_LONG_SIDE", 1024))
DEBUG = bool(globals().get("DEBUG", False))
BODY_FACE_POLICY = str(globals().get("BODY_FACE_POLICY", "largest"))
BODY_FACE_INDEX = int(globals().get("BODY_FACE_INDEX", 0))
IDENTITY_THRESH = float(globals().get("IDENTITY_THRESH", 0.35))
BODY_PSNR_THRESH = float(globals().get("BODY_PSNR_THRESH", 28.0))

os.environ.setdefault("REFACE_ROOT", "/content/REFace")
cfg = load_config(REPO / "configs" / "reface.yaml")
cfg.update({
    "reface_root": os.environ["REFACE_ROOT"],
    "output_long_side": OUTPUT_LONG_SIDE,
    "body_face_policy": BODY_FACE_POLICY,
    "body_face_index": BODY_FACE_INDEX,
    "ddim_steps": STEPS,
    "guidance_scale": CFG,
    "seed": SEED,
    "save_debug": DEBUG,
    "verbose": DEBUG,
})

body = Image.open(REPO / "data" / "custom" / "body.png").convert("RGB")
face = Image.open(REPO / "data" / "custom" / "face.png").convert("RGB")
run_dir = colab_demo.make_run_dir("/content/headswap_outputs")

try:
    pipe = create_pipeline(cfg)
    result = pipe.run(body, face, out_dir=run_dir)
    out = result.image
    latency = float(result.latency_s)
    quality = colab_demo.score_result(
        body=body, face=face, result=out, latency_s=latency,
        cache_dir=REPO / ".cache" / "headswap_v2",
        pipeline="reface", identity_thresh=IDENTITY_THRESH,
        body_psnr_thresh=BODY_PSNR_THRESH, stitch=True,
    )
    package = colab_demo.save_output_package(
        run_dir,
        result_image=out,
        run_config={
            "pipeline": "reface",
            "knobs": {
                "SEED": SEED, "STEPS": STEPS, "CFG": CFG,
                "OUTPUT_LONG_SIDE": OUTPUT_LONG_SIDE, "DEBUG": DEBUG,
                "BODY_FACE_POLICY": BODY_FACE_POLICY, "BODY_FACE_INDEX": BODY_FACE_INDEX,
            },
            "meta": dict(result.meta or {}),
            "latency_s": latency,
        },
        metrics=quality,
        timing={"total_s": latency, "sampling_s": latency},
        debug_paths=result.debug_paths,
        save_debug=DEBUG,
    )
    RESULT_IMAGE = out
    RUN_DIR = run_dir
    QUALITY = quality
    LATENCY_S = latency
    print(f"✓ Done in {latency:.1f}s → {run_dir}")
    colab_demo.print_quality_report(quality)
except Exception as exc:
    colab_demo.fail(f"REFace run failed: {exc}")
    if DEBUG:
        traceback.print_exc()
    raise RuntimeError(str(exc)) from exc


---
## 6 · Results


In [ ]:
from IPython.display import display, Markdown
from PIL import Image
from pathlib import Path
import importlib.util

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

body = Image.open(REPO / "data" / "custom" / "body.png").convert("RGB")
face = Image.open(REPO / "data" / "custom" / "face.png").convert("RGB")
result = globals().get("RESULT_IMAGE")
if result is None:
    raise SystemExit("No RESULT_IMAGE — run §5 first.")
display(Markdown("### Side-by-side"))
display(colab_demo.show_side_by_side(body, face, result))
display(Markdown("### Full result"))
display(result)
print(f"Run dir: {globals().get('RUN_DIR')}")
print("Stable: /content/headswap_outputs/HEADSWAP_RESULT.png")


---
## 7 · Run summary


In [ ]:
import importlib.util
from pathlib import Path

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

result = globals().get("RESULT_IMAGE")
run_dir = globals().get("RUN_DIR")
latency = float(globals().get("LATENCY_S") or 0)
quality = globals().get("QUALITY") or {}
colab_demo.print_run_summary(
    success=result is not None,
    total_s=latency,
    sampling_s=latency,
    steps=int(globals().get("STEPS") or 50),
    seed=int(globals().get("SEED") or 42),
    gpu=None,
    resolution=list(result.size) if result is not None else None,
    output_path=(Path(run_dir) / "result.png") if run_dir else None,
    quality=quality,
)
print(f"  policy            {globals().get('BODY_FACE_POLICY')}")
print(f"  cfg/scale         {globals().get('CFG')}")


---
## Notes

| Item | Detail |
| --- | --- |
| Model | REFace diffusion face-swap (WACV 2025) |
| License | Non-commercial research (CelebAMask-HQ trained) |
| Multi-person | Crops selected face window, swaps, pastes back |
| Outputs | `/content/headswap_outputs/run_*/` |
| Upstream | https://github.com/Sanoojan/REFace |

Ethics: only use with consent.
